In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import joblib
import os

In [2]:
# Load the Raw Dataset
csv_path = '../../data/Synthetic_Financial_datasets_log.csv'
df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.dtypes)
print(df.head())

Dataset shape: (6362620, 11)
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isF

In [3]:
# Inspect and Clean the Data
# Basic type normalization
for col in ['type', 'nameOrig', 'nameDest']:
    df[col] = df[col].astype(str)

print("Data types after normalization:")
print(df.dtypes)
print(f"\nMissing values:\n{df.isnull().sum()}")

Data types after normalization:
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [4]:
# Feature Engineering and Column Selection
# Capture suspicious balance movement + anonymize IDs via prefixes
df['nameOrig_prefix'] = df['nameOrig'].str[0] 
df['nameDest_prefix'] = df['nameDest'].str[0]

df['orig_balance_delta'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['dest_balance_delta'] = df['newbalanceDest'] - df['oldbalanceDest']

# Avoid dividing by 0
old_org_denom = df['oldbalanceOrg'].replace(0, np.nan).fillna(1)
new_org_denom = df['newbalanceOrig'].replace(0, np.nan).fillna(1)

df['amount_to_oldOrg'] = df['amount'] / old_org_denom
df['amount_to_newOrg'] = df['amount'] / new_org_denom

print("Feature engineering completed")
print(f"New columns added: {[col for col in df.columns if col not in ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']]}")

Feature engineering completed
New columns added: ['nameOrig_prefix', 'nameDest_prefix', 'orig_balance_delta', 'dest_balance_delta', 'amount_to_oldOrg', 'amount_to_newOrg']


In [5]:
# Handle Missing Values and Data Types
# Check for any remaining missing values
print(f"Missing values after feature engineering:\n{df.isnull().sum()}")

# Ensure target is integer
df['isFraud'] = df['isFraud'].astype(int)

print("Data types finalized:")
print(df.dtypes)

Missing values after feature engineering:
step                  0
type                  0
amount                0
nameOrig              0
oldbalanceOrg         0
newbalanceOrig        0
nameDest              0
oldbalanceDest        0
newbalanceDest        0
isFraud               0
isFlaggedFraud        0
nameOrig_prefix       0
nameDest_prefix       0
orig_balance_delta    0
dest_balance_delta    0
amount_to_oldOrg      0
amount_to_newOrg      0
dtype: int64
Data types finalized:
step                    int64
type                   object
amount                float64
nameOrig               object
oldbalanceOrg         float64
newbalanceOrig        float64
nameDest               object
oldbalanceDest        float64
newbalanceDest        float64
isFraud                 int64
isFlaggedFraud          int64
nameOrig_prefix        object
nameDest_prefix        object
orig_balance_delta    float64
dest_balance_delta    float64
amount_to_oldOrg      float64
amount_to_newOrg      float64
dtype

In [6]:
# Define target and features
y = df['isFraud']

feature_columns = [
    'step',
    'type',
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest',
    'nameOrig_prefix',
    'nameDest_prefix',
    'orig_balance_delta',
    'dest_balance_delta',
    'amount_to_oldOrg',
    'amount_to_newOrg',
]

X = df[feature_columns]

print(f"Target shape: {y.shape}")
print(f"Features shape: {X.shape}")
print(f"Feature columns: {feature_columns}")

Target shape: (6362620,)
Features shape: (6362620, 13)
Feature columns: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'nameOrig_prefix', 'nameDest_prefix', 'orig_balance_delta', 'dest_balance_delta', 'amount_to_oldOrg', 'amount_to_newOrg']


In [7]:
# Encode Categorical Variables
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical features: {categorical_features}")

# Initialize encoder
cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit and transform categorical features
X_cat_encoded = cat_encoder.fit_transform(X[categorical_features])

print(f"Categorical features encoded. Shape: {X_cat_encoded.shape}")

Categorical features: ['type', 'nameOrig_prefix', 'nameDest_prefix']
Categorical features encoded. Shape: (6362620, 3)


In [8]:
# Normalize Numeric Features
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Numeric features: {numeric_features}")

# Initialize scaler
scaler = StandardScaler()

# Fit and transform numeric features
X_num_scaled = scaler.fit_transform(X[numeric_features])

print(f"Numeric features scaled. Shape: {X_num_scaled.shape}")

Numeric features: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'orig_balance_delta', 'dest_balance_delta', 'amount_to_oldOrg', 'amount_to_newOrg']
Numeric features scaled. Shape: (6362620, 10)


In [9]:
# Combine preprocessed features
# Create column names for preprocessed data
preprocessed_columns = numeric_features + categorical_features

# Combine scaled numeric and encoded categorical
X_preprocessed = np.concatenate([X_num_scaled, X_cat_encoded], axis=1)

print(f"Combined preprocessed features shape: {X_preprocessed.shape}")
print(f"Columns: {preprocessed_columns}")

Combined preprocessed features shape: (6362620, 13)
Columns: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'orig_balance_delta', 'dest_balance_delta', 'amount_to_oldOrg', 'amount_to_newOrg', 'type', 'nameOrig_prefix', 'nameDest_prefix']


In [12]:
# Create final preprocessed dataframe
preprocessed_df = pd.DataFrame(X_preprocessed, columns=preprocessed_columns)
preprocessed_df['isFraud'] = y.values
preprocessed_df['isFlaggedFraud'] = df['isFlaggedFraud'].values

print(f"Final preprocessed dataframe shape: {preprocessed_df.shape}")
print(preprocessed_df.head())

Final preprocessed dataframe shape: (6362620, 15)
       step    amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
0 -1.703042 -0.281560      -0.229810       -0.237622       -0.323814   
1 -1.703042 -0.294767      -0.281359       -0.285812       -0.323814   
2 -1.703042 -0.297555      -0.288654       -0.292442       -0.323814   
3 -1.703042 -0.297555      -0.288654       -0.292442       -0.317582   
4 -1.703042 -0.278532      -0.274329       -0.282221       -0.323814   

   newbalanceDest  orig_balance_delta  dest_balance_delta  amount_to_oldOrg  \
0       -0.333411           -0.211876           -0.152896         -0.139024   
1       -0.333411           -0.157490           -0.152896         -0.139024   
2       -0.333411           -0.146011           -0.152896         -0.139022   
3       -0.333411           -0.146011           -0.178952         -0.139022   
4       -0.333411           -0.224345           -0.152896         -0.139023   

   amount_to_newOrg  type  nameOrig_prefix

In [13]:
# Save Preprocessed Data
output_path = 'preprocessed_data.csv'
preprocessed_df.to_csv(output_path, index=False)
print(f"Preprocessed data saved to: {output_path}")

# Also save the preprocessing objects for later use
preprocessing_artifacts = {
    'scaler': scaler,
    'cat_encoder': cat_encoder,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'feature_columns': feature_columns
}

joblib.dump(preprocessing_artifacts, 'preprocessing_artifacts.pkl')
print("Preprocessing artifacts saved to: preprocessing_artifacts.pkl")

Preprocessed data saved to: preprocessed_data.csv
Preprocessing artifacts saved to: preprocessing_artifacts.pkl
